### Validate MAP message

In [3]:
from v2x_msg_validator.validator import V2XMessageValidator

validator = V2XMessageValidator(revision="j2735_202409")

In [10]:
import yaml
from pprint import pprint

def read_sample_yaml(filepath):
    with open(filepath, "r") as file:
        data = yaml.safe_load(file)
    return data

data = read_sample_yaml("../samples/sampleMAP.yaml")
# keep a subset of intersections
data["value"]["MapData"]["intersections"] = data["value"]["MapData"]["intersections"][:1]
pprint(data)

{'messageId': 18,
 'value': {'MapData': {'intersections': [{'id': {'id': 156},
                                          'laneSet': [{'connectsTo': [{'connectingLane': {'lane': 25,
                                                                                          'maneuver': '8000'},
                                                                       'signalGroup': 2},
                                                                      {'connectingLane': {'lane': 29,
                                                                                          'maneuver': '2000'},
                                                                       'signalGroup': 2}],
                                                       'laneAttributes': {'directionalUse': '80',
                                                                          'laneType': {'vehicle': '00'},
                                                                          'sharedWith': '0000'},
              

In the MAP message above, several fields are problematic, e.g., `'directionalUse': '80'`, `'maneuver': '8000'`. The `directionalUse` is a Bit String of size 2, and thus should take value from `00` to `11` in binary format. Similarly, `maneuver` is a Bit String of size 12, but the data have 16 bits. The same issue happens with `sharedWith`, which only has 10 bits.  These are expected to lead to validation errors.

In [11]:
errors = validator.validate(data)
pprint(errors)

[ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].laneAttributes.directionalUse', msg='value out of size constraint', val=(128, 8)],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].laneAttributes.sharedWith', msg='value out of size constraint', val=(0, 4)],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].maneuvers', msg='value out of size constraint', val=(40960, 16)],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].connectsTo[0].connectingLane.maneuver', msg='value out of size constraint', val=(32768, 16)],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[0].connectsTo[1].connectingLane.maneuver', msg='value out of size constraint', val=(8192, 16)],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[1].laneAttributes.directionalUse', msg='value out of size constraint', val=(128, 8)],
 ASN1ObjValErr[key='MapData.intersections[0].laneSet[1].laneAttributes.sharedWith', msg='value out of size constraint', val=(0, 4)],
 ASN1ObjValErr[key='MapData.interse

Note the feature of the tool is that it will point to all fields that are invalid, including the index within any list and the nested keys. A descriptive error message is also provided.

### Validate SPaT message

In [16]:
data = read_sample_yaml("../samples/sampleSPAT.yaml")
pprint(data)

{'messageId': 19,
 'value': {'SPAT': {'intersections': [{'id': {'id': 9709},
                                       'moy': 428788,
                                       'name': 'West Intersection',
                                       'revision': 1,
                                       'states': [{'signalGroup': 1,
                                                   'state-time-speed': [{'eventState': 'stop-And-Remain',
                                                                         'timing': {'minEndTime': 17261}}]},
                                                  {'signalGroup': 2,
                                                   'state-time-speed': [{'eventState': 'protected-Movement-Allowed',
                                                                         'timing': {'maxEndTime': 17399,
                                                                                    'minEndTime': 17399}}]},
                                                  {'signalGroup

In [18]:
errors = validator.validate(data)
pprint(errors)

[]


### Validate TravelerInformation message

In [9]:
data = read_sample_yaml("../samples/sampleTIM.yaml")
pprint(data)

{'messageId': 31,
 'value': {'TravelerInformation': {'dataFrames': [{'content': {'speedLimit': [{'item': {'itis': 27}},
                                                                              {'item': {'text': 'Curve '
                                                                                                'Ahead'}},
                                                                              {'item': {'itis': 2564}},
                                                                              {'item': {'text': '25'}},
                                                                              {'item': {'itis': 8720}}]},
                                                   'contentNew': {'frictionInfo': {'roadSurfaceDescription': {'portlandCement': {'type': 'newSharp'}}}},
                                                   'doNotUse1': 0,
                                                   'doNotUse2': 0,
                                                   'doNotUse3': 0,


In [10]:
errors = validator.validate(data)
pprint(errors)

[]


In [11]:
import json
from v2x_codecs import j2735_202409
TIM = j2735_202409.TravelerInformation.TravelerInformation
# convert to uper encoding, decode, and compare against original data
buf = TIM.to_uper()

print("==== uper encoding ===")
pprint(buf)

# decode from buffer
TIM.from_uper(buf)

# convert to json
data_decoded = json.loads(TIM.to_json())

print("==== decoded json ===")
pprint(data_decoded)


assert data["value"]["TravelerInformation"]==data_decoded
print("Decoded data match initial data")

==== uper encoding ===
(b'\x00\x11\x00b\x99\xb9\xea\xc2z\x9b\xaat#!X \xc0\xa9\xdf@(\x00~S7=XOSuN\x84'
 b'd\x00\xb7!\x00\x00\x00\xb8\xf57N6f\xe7\xacP\x13\xec\xe3\xd4\xdd\xc1\t\x9b'
 b'\x9e\x98\x80PS\x8fSx\xf9fnzZ\x81A@\x03@\x00\xde\xa1\xf5\xe5\xdb*\x08:2'
 b'\xe1\xc8\n\x04\x8b&\xa2!\x00\x10 \x00\x00')
==== decoded json ===
{'dataFrames': [{'content': {'speedLimit': [{'item': {'itis': 27}},
                                            {'item': {'text': 'Curve Ahead'}},
                                            {'item': {'itis': 2564}},
                                            {'item': {'text': '25'}},
                                            {'item': {'itis': 8720}}]},
                 'contentNew': {'frictionInfo': {'roadSurfaceDescription': {'portlandCement': {'type': 'newSharp'}}}},
                 'doNotUse1': 0,
                 'doNotUse2': 0,
                 'doNotUse3': 0,
                 'doNotUse4': 0,
                 'durationTime': 32000,
                 'frameTyp

### Validate PersonalSafetyMessage

In [12]:
data = read_sample_yaml("../samples/samplePSM.yaml")
pprint(data)

{'messageId': 32,
 'value': {'PersonalSafetyMessage': {'accelSet': {'lat': 0,
                                                  'long': 0,
                                                  'vert': 0,
                                                  'yaw': 0},
                                     'accuracy': {'orientation': 0,
                                                  'semiMajor': 50,
                                                  'semiMinor': 50},
                                     'basicType': 'aPEDESTRIAN',
                                     'heading': 8000,
                                     'id': '00000001',
                                     'msgCnt': 1,
                                     'pathPrediction': {'confidence': 200,
                                                        'radiusOfCurve': 32767},
                                     'position': {'elevation': 400,
                                                  'lat': 389549153,
                    

In [13]:
errors = validator.validate(data)
pprint(errors)

[]


In [14]:
from v2x_codecs import j2735_202409
PSM = j2735_202409.PersonalSafetyMessage.PersonalSafetyMessage
# convert to uper encoding, decode, and compare against original data
buf = PSM.to_uper()

print("==== uper encoding ===")
pprint(buf)

# decode from buffer
PSM.from_uper(buf)

# convert to json
data_decoded = json.loads(PSM.to_json())

print("==== decoded json ===")
pprint(data_decoded)

assert data["value"]["PersonalSafetyMessage"]==data_decoded
print("Decoded data match initial data")

==== uper encoding ===
(b'X\x00\x03_\x90\x04\x00\x00\x00\x05L\xdc\xf5a=M\xd5:\x11\x9022\x00\x00'
 b'\x03\xe9\xf4\x07\xd0}\x07\xf7\xff\xf7\xff\xf6@ ')
==== decoded json ===
{'accelSet': {'lat': 0, 'long': 0, 'vert': 0, 'yaw': 0},
 'accuracy': {'orientation': 0, 'semiMajor': 50, 'semiMinor': 50},
 'basicType': 'aPEDESTRIAN',
 'heading': 8000,
 'id': '00000001',
 'msgCnt': 1,
 'pathPrediction': {'confidence': 200, 'radiusOfCurve': 32767},
 'position': {'elevation': 400, 'lat': 389549153, 'long': -771488965},
 'propulsion': {'human': 'onFoot'},
 'secMark': 45000,
 'speed': 125}
Decoded data match initial data
